**Import necessary libraries**

In [2]:
pip install gensim

   ---------------------------------------- 0.0/24.0 MB ? eta -:--:--
    --------------------------------------- 0.3/24.0 MB 9.9 MB/s eta 0:00:03
    --------------------------------------- 0.6/24.0 MB 5.8 MB/s eta 0:00:05
   - -------------------------------------- 0.7/24.0 MB 5.6 MB/s eta 0:00:05
   - -------------------------------------- 0.9/24.0 MB 4.9 MB/s eta 0:00:05
   - -------------------------------------- 1.1/24.0 MB 4.5 MB/s eta 0:00:06
   - -------------------------------------- 1.2/24.0 MB 4.4 MB/s eta 0:00:06
   -- ------------------------------------- 1.4/24.0 MB 4.2 MB/s eta 0:00:06
   -- ------------------------------------- 1.6/24.0 MB 4.0 MB/s eta 0:00:06
   -- ------------------------------------- 1.7/24.0 MB 3.9 MB/s eta 0:00:06
   --- ------------------------------------ 1.8/24.0 MB 3.7 MB/s eta 0:00:06
   --- ------------------------------------ 1.9/24.0 MB 3.6 MB/s eta 0:00:07
   --- ------------------------------------ 2.0/24.0 MB 3.5 MB/s eta 0:00:07
   ---


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: C:\Users\AKHIL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
pip install xgboost

  Using cached xgboost-2.1.1-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-2.1.1-py3-none-win_amd64.whl (124.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: C:\Users\AKHIL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: C:\Users\AKHIL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

**Load the Dataset**

In [10]:
# Load the dataset from a CSV file
file_path = r"C:\Users\AKHIL\research\dataset\cleaned_data1.csv"
dataset = pd.read_csv(file_path)

**Text Preprocessing Function**

In [11]:
# Define a function to preprocess text data
def preprocess_text(text):
    text = re.sub(r'\W', ' ', text)  # Remove all non-word characters
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = text.lower()  # Convert to lowercase
    return text
# Apply the text preprocessing function to the 'text' column of the dataset
dataset['text'] = dataset['text'].apply(preprocess_text)

**Train Word2Vec Model**

In [12]:
# Tokenize the preprocessed text
tokenized_text = dataset['text'].apply(lambda x: x.split())

# Train a Word2Vec model on the tokenized text
w2v_model = Word2Vec(sentences=tokenized_text, vector_size=100, window=5, min_count=1, workers=4)


**Create Function to Get Average Word2Vec Vectors and to vectorize text data**

In [13]:
# Train a Word2Vec model on the tokenized text
w2v_model = Word2Vec(sentences=tokenized_text, vector_size=100, window=5, min_count=1, workers=4)

# Define a function to get the average Word2Vec vector for a list of words
def get_avg_w2v_vector(words, model, vector_size):
    feature_vec = np.zeros((vector_size,), dtype='float32')
    n_words = 0
    for word in words:
        if word in model.wv:
            n_words += 1
            feature_vec = np.add(feature_vec, model.wv[word])
    if n_words > 0:
        feature_vec = np.divide(feature_vec, n_words)
    return feature_vec

# Convert the tokenized text to Word2Vec vectors
vector_size = 100
dataset['w2v_vector'] = tokenized_text.apply(lambda x: get_avg_w2v_vector(x, w2v_model, vector_size))


**Convert Vectors to Format Suitable for XGBoost**

In [14]:
# Convert the Word2Vec vectors to a NumPy array
X = np.array(dataset['w2v_vector'].tolist())
y = dataset['label']

# Encode the labels using LabelEncoder
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

**Train-Test Split**

In [15]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost model on the training data
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train)

C:\Users\AKHIL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\xgboost\core.py:158: UserWarning: [22:27:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, objective='multi:softprob', ...)

**Make Predictions and Evaluate Model**

In [16]:
# Make predictions on the testing data
y_pred = xgb_model.predict(X_test)

# Evaluate the model using classification report
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

               precision    recall  f1-score   support

     business       0.76      0.63      0.69       104
entertainment       0.70      0.67      0.68        75
     politics       0.66      0.75      0.70        80
       sports       0.85      0.86      0.86       100
         tech       0.66      0.75      0.70        68

     accuracy                           0.73       427
    macro avg       0.73      0.73      0.73       427
 weighted avg       0.74      0.73      0.73       427



In [ ]:
dataset.head()

,text,label,w2v_vector
0,ad sales boost time warner profit quarterly pr...,business,"[0.15717256, 0.51453424, 0.08027557, -0.048726..."
1,dollar gains on greenspan speech the dollar ha...,business,"[0.19153003, 0.5771954, 0.07917573, -0.0256851..."
2,yukos unit buyer faces loan claim the owners o...,business,"[0.1828429, 0.57953286, 0.21261422, -0.0100212..."
3,high fuel prices hit bas profits british airwa...,business,"[0.16046473, 0.47035736, 0.028004767, -0.06990..."
4,pernod takeover talk lifts domecq shares in uk...,business,"[0.14263614, 0.41045335, 0.07699512, -0.039815..."


In [9]:
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# Load the dataset from a CSV file
file_path = 'cleaned_data1.csv'
dataset = pd.read_csv(file_path)

# Define a function to preprocess text data
def preprocess_text(text):
    text = re.sub(r'\W', ' ', text)  # Remove all non-word characters
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = text.lower()  # Convert to lowercase
    return text

# Apply the text preprocessing function to the 'text' column of the dataset
dataset['text'] = dataset['text'].apply(preprocess_text)

# Tokenize the preprocessed text
tokenized_text = dataset['text'].apply(lambda x: x.split())

# Train a Word2Vec model on the tokenized text
w2v_model = Word2Vec(sentences=tokenized_text, vector_size=100, window=5, min_count=1, workers=4)

# Define a function to get the average Word2Vec vector for a list of words
def get_avg_w2v_vector(words, model, vector_size):
    feature_vec = np.zeros((vector_size,), dtype='float32')
    n_words = 0
    for word in words:
        if word in model.wv:
            n_words += 1
            feature_vec = np.add(feature_vec, model.wv[word])
    if n_words > 0:
        feature_vec = np.divide(feature_vec, n_words)
    return feature_vec

# Convert the tokenized text to Word2Vec vectors
vector_size = 100
dataset['w2v_vector'] = tokenized_text.apply(lambda x: get_avg_w2v_vector(x, w2v_model, vector_size))

# Convert the Word2Vec vectors to a NumPy array
X = np.array(dataset['w2v_vector'].tolist())
y = dataset['label']

# Encode the labels using LabelEncoder
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train an XGBoost model on the training data
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = xgb_model.predict(X_test)

# Evaluate the model using classification report
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


               precision    recall  f1-score   support

     business       0.78      0.66      0.72       104
entertainment       0.72      0.68      0.70        75
     politics       0.66      0.80      0.72        80
       sports       0.90      0.90      0.90       100
         tech       0.73      0.76      0.75        68

     accuracy                           0.76       427
    macro avg       0.76      0.76      0.76       427
 weighted avg       0.77      0.76      0.76       427



In [18]:
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk

# # Download required NLTK data files
# nltk.download('punkt')
# nltk.download('wordnet')

# Load the dataset from a CSV file
file_path = r"C:\Users\AKHIL\research\dataset\cleaned_data1.csv"
dataset = pd.read_csv(file_path)

# Define a function to preprocess text data
def preprocess_text(text):
    text = re.sub(r'\W', ' ', text)  # Remove all non-word characters
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = text.lower()  # Convert to lowercase
    return text

# Apply the text preprocessing function to the 'text' column of the dataset
dataset['text'] = dataset['text'].apply(preprocess_text)

# Initialize the WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

# Define a function to perform lemmatization
def lemmatize_text(text):
    words = word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(lemmatized_words)

# Apply the lemmatization function to the 'text' column of the dataset
dataset['text'] = dataset['text'].apply(lemmatize_text)

# Tokenize the lemmatized text
tokenized_text = dataset['text'].apply(lambda x: x.split())

# Train a Word2Vec model on the tokenized text
w2v_model = Word2Vec(sentences=tokenized_text, vector_size=100, window=5, min_count=1, workers=4)

# Define a function to get the average Word2Vec vector for a list of words
def get_avg_w2v_vector(words, model, vector_size):
    feature_vec = np.zeros((vector_size,), dtype='float32')
    n_words = 0
    for word in words:
        if word in model.wv:
            n_words += 1
            feature_vec = np.add(feature_vec, model.wv[word])
    if n_words > 0:
        feature_vec = np.divide(feature_vec, n_words)
    return feature_vec

# Convert the tokenized text to Word2Vec vectors
vector_size = 100
dataset['w2v_vector'] = tokenized_text.apply(lambda x: get_avg_w2v_vector(x, w2v_model, vector_size))

# Convert the Word2Vec vectors to a NumPy array
X = np.array(dataset['w2v_vector'].tolist())
y = dataset['label']

# Encode the labels using LabelEncoder
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=60)

# Train an XGBoost model on the training data
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = xgb_model.predict(X_test)

# Evaluate the model using classification report
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))



C:\Users\AKHIL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\xgboost\core.py:158: UserWarning: [22:30:49] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


               precision    recall  f1-score   support

     business       0.81      0.79      0.80       138
entertainment       0.79      0.72      0.76       122
     politics       0.74      0.79      0.76       115
       sports       0.87      0.86      0.87       162
         tech       0.78      0.85      0.81       104

     accuracy                           0.80       641
    macro avg       0.80      0.80      0.80       641
 weighted avg       0.80      0.80      0.80       641



In [14]:
from sklearn.metrics import confusion_matrix
# Calculate and print the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

Confusion Matrix:
[[110   4   9   3  12]
 [  6  87   6  17   6]
 [ 11   1  91   4   8]
 [  0  11   4 145   2]
 [ 11   4   6   1  82]]
